# Colab 전환 노트북 — vLLM으로 디베이트 품질 실험

로컬(Ollama/Qwen3 8B)에서 배관을 확인한 뒤, 같은 코드를 **모델만 바꿔** Colab에서 돌린다.
코드 변경은 없다. `FDM_BACKEND=vllm` 과 모델 이름만 환경변수로 지정한다.

| 런타임 | 권장 모델 배치 |
|---|---|
| 무료 T4 (16GB) | 토론·심판 모두 `Qwen/Qwen3-14B-AWQ` 또는 `google/gemma-3-12b-it` (4bit) |
| L4 / A100 | 토론 `Qwen/Qwen3-14B-AWQ`, 심판 `LGAI-EXAONE/EXAONE-4.0-32B` (4bit) ← 목표 품질 |

심판 모델을 크게 쓰는 이유는 최종 적합성 판정 품질이 결과를 좌우하기 때문이다(비대칭 배치).
단일 vLLM 서버는 모델 하나만 서빙하므로, 비대칭 배치를 쓰려면 아래 **B안**처럼 서버를 두 개 띄우거나
1차 실험은 동일 모델로 시작한다.

In [ ]:
!nvidia-smi
import torch, os
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU 없음 — 런타임 유형을 GPU로 변경')

## 1. 프로젝트 준비

GitHub에 올렸다면 clone, 아니면 좌측 파일 탭에 프로젝트 zip을 올려 압축을 푼다.

In [ ]:
# A) 저장소에서 가져오기
# !git clone https://github.com/<user>/<repo>.git fdm_project

# B) zip 업로드 후
# !unzip -q -O utf-8 '/content/금융 디베이트 모델.zip' -d /content/fdm_project

%cd /content/fdm_project
!ls

In [ ]:
!pip -q install vllm==0.11.0 pydantic httpx pandas typer rich python-dotenv
!pip -q install -e .  # fdm 패키지 설치 (src 레이아웃)

## 2. Nemotron-Personas-Korea 내려받아 로컬 캐시로 저장

한 번 저장해두면 이후 실행은 오프라인으로 재현된다.

In [ ]:
!pip -q install datasets pyarrow
!python scripts/fetch_personas.py --limit 5000
!ls -la data/personas/

## 3. vLLM 서버 기동

디베이트는 상품 1건당 LLM 호출이 수십~수백 회이므로 처리량이 중요하다.
`--max-num-seqs` 를 올리고 코드에서는 `workers` 를 함께 올려 병렬로 밀어넣는다.

In [ ]:
import os, subprocess, time, httpx

MODEL = 'Qwen/Qwen3-14B-AWQ'          # T4/L4 공용 출발점
# MODEL = 'LGAI-EXAONE/EXAONE-4.0-32B'  # L4/A100 + 4bit 양자화 필요

log = open('/content/vllm.log', 'w')
proc = subprocess.Popen(
    ['vllm', 'serve', MODEL,
     '--port', '8000',
     '--max-model-len', '8192',
     '--max-num-seqs', '16',
     '--gpu-memory-utilization', '0.90',
     '--dtype', 'auto'],
    stdout=log, stderr=subprocess.STDOUT,
)

for i in range(120):  # 모델 다운로드 포함 최대 20분 대기
    try:
        if httpx.get('http://localhost:8000/v1/models', timeout=3).status_code == 200:
            print('vLLM 준비 완료'); break
    except Exception:
        pass
    time.sleep(10)
else:
    print('기동 실패 — /content/vllm.log 확인')
    print(open('/content/vllm.log').read()[-3000:])

In [ ]:
os.environ['FDM_BACKEND'] = 'vllm'
os.environ['FDM_VLLM_BASE_URL'] = 'http://localhost:8000/v1'
os.environ['FDM_MODEL_SMALL'] = MODEL
os.environ['FDM_MODEL_JUDGE'] = MODEL  # 비대칭 배치 시 두 번째 서버(포트 8001) 모델명으로 교체
os.environ['FDM_MAX_TOKENS'] = '1200'

!python -m fdm.cli doctor

### (선택) 비대칭 배치 — 심판만 큰 모델

A100 40GB 이상에서 두 서버를 동시에 띄울 때만 쓴다. `LLMClient`는 역할별로 base_url이 아니라
모델명만 분기하므로, 서버를 2개 쓰려면 vLLM 앞에 라우팅 프록시를 두거나
`src/fdm/llm.py`의 `base_url` 을 역할별로 분기하도록 3줄만 수정하면 된다.

## 4. 디베이트 1건 확인 (프롬프트 품질 눈으로 검증)

In [ ]:
!python -m fdm.cli debate 01_youth_step_saving --full

## 5. 전체 시뮬레이션 + 민감도 + 애블레이션

호출량이 많다. T4에서는 `--seeds 3 --personas-per-segment 3` 정도로 시작한다.

In [ ]:
!python -m fdm.cli simulate 01_youth_step_saving \
    --seeds 3 --personas-per-segment 3 --workers 8 \
    --with-sensitivity --with-ablation

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('outputs/report_SV-2026-001.md', encoding='utf-8').read()))

## 6. 애블레이션만 반복 실행 (핵심 실증 수치)

심사에서 가장 중요한 숫자다. 시드를 늘려 분산을 줄이고, 정답셋을 실제 조정결정례로
교체한 뒤 다시 측정한다.

In [ ]:
!python -m fdm.cli ablation --seeds 3

In [ ]:
# 여러 상품 일괄 실행
import glob, pathlib
for f in sorted(glob.glob('data/products/*.json')):
    stem = pathlib.Path(f).stem
    print('=' * 20, stem)
    !python -m fdm.cli simulate {stem} --seeds 3 --personas-per-segment 3 --workers 8

In [ ]:
# 결과 zip으로 내려받기
!zip -qr /content/outputs.zip outputs/
from google.colab import files
files.download('/content/outputs.zip')

In [ ]:
proc.terminate()  # 서버 종료